In [89]:
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

import html
import time
from bs4 import BeautifulSoup
import re
import pandas as pd
import os
from dotenv import load_dotenv
import time
from IPython.display import clear_output
import traceback

import os
import json
import requests
from urllib.parse import urljoin

BASE_URL = "https://www.lamoncloa.gob.es"
URL_INDICE = "https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23F.aspx"

HTML_DIR = "./html"
PDF_BASE_DIR = "./pdf"
JSON_OUTPUT_PATH = "./json/listado_de_archivos.json"

# Carga del sitio web principal con Selenium

Descarga del driver más reciente de 'Chrome for Testing':

https://googlechromelabs.github.io/chrome-for-testing/

Inicialización del driver

In [108]:
load_dotenv("../../apis/.env")
chromedriver_path = os.getenv("CHROMEDRIVER_PATH")

service = Service(executable_path=chromedriver_path)
options = webdriver.ChromeOptions()

# Configuramos las preferencias de descarga de Chrome para guardar directamente los PDFs
prefs = {
    "download.default_directory": os.path.abspath(PDF_BASE_DIR),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True,
    "safebrowsing.enabled": True,
    "safebrowsing.disable_download_protection": True,
    "profile.default_content_setting_values.automatic_downloads": 1
}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(service=service, options=options)
# driver = uc.Chrome(headless=False) #### USING UNDETECTED CHROMEDRIVER

Carga de la página principal

In [109]:
driver.implicitly_wait(5)
driver.get(URL_INDICE)

time.sleep(0.5)
try:
    accept_cookies_button = driver.find_element(
        By.XPATH, '//*[@id="ctl00_CookieInfo_btnLink"]'
    )
    accept_cookies_button.click()
except:
    pass

## Descarga del contenido del sitio web principal

In [97]:
dd23f_html = driver.page_source
print(dd23f_html[:100],"\n...")
print(dd23f_html[-100:])
# today = pd.Timestamp.today().strftime('%Y_%m_%d')
with open(os.path.join(HTML_DIR, 'dd23f_html_source.html'), 'w', encoding='utf-8') as file:
    file.write(dd23f_html)

<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="es" lang="es" class="no-js"><head id="Head"><me 
...
ookies','event_label': currentDate + ' [' + window.location.pathname + ']'});</script></body></html>


In [98]:
HTML_PATH = os.path.join(HTML_DIR, 'dd23f_html_source.html')
with open(HTML_PATH, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

In [99]:
structure = {}

# Identify main <h1>
main_h1 = soup.find("h1")
if not main_h1:
    raise ValueError("No se encontró <h1> en la página.")

# Iterate over each documents-box
for documents_box in main_h1.find_all_next("div", class_="documents-box"):

    h2 = documents_box.find("h2")
    if not h2:
        continue

    level_1_name = h2.get_text(strip=True)
    structure[level_1_name] = {}

    docs_list = documents_box.find("ul", class_="docs-list")
    if not docs_list:
        continue

    # Only first-level <li> inside docs-list
    for li in docs_list.find_all("li", recursive=False):

        h3 = li.find("h3")

        # Case 1: Normal structure with h3
        if h3:
            level_2_name = h3.get_text(strip=True)
            structure[level_1_name][level_2_name] = []

            for link in li.find_all("a", href=True):
                href = link["href"]
                if href.lower().endswith(".pdf"):
                    full_url = urljoin(BASE_URL, href)
                    file_name = link.get_text(strip=True)

                    structure[level_1_name][level_2_name].append({
                        "file_name": file_name,
                        "url": full_url
                    })

        # Case 2: PDFs directly under ministry (no h3)
        else:
            default_category = "Documents"
            if default_category not in structure[level_1_name]:
                structure[level_1_name][default_category] = []

            for link in li.find_all("a", href=True):
                href = link["href"]
                if href.lower().endswith(".pdf"):
                    full_url = urljoin(BASE_URL, href)
                    file_name = link.get_text(strip=True)

                    structure[level_1_name][default_category].append({
                        "file_name": file_name,
                        "url": full_url
                    })

# Save JSON
os.makedirs(os.path.dirname(JSON_OUTPUT_PATH), exist_ok=True)

with open(JSON_OUTPUT_PATH, "w", encoding="utf-8") as jf:
    json.dump(structure, jf, indent=4, ensure_ascii=False)

print("Listado de archivos guardado en:", JSON_OUTPUT_PATH)

Listado de archivos guardado en: ./json/listado_de_archivos.json


## Descarga de los PDFs

In [110]:
MAX_NAME_LENGTH = 50
DOWNLOAD_TIMEOUT = 30

def sanitize_name(name):
    name = re.sub(r'[\\/*?:"<>|]', "", name).strip()
    if len(name) > MAX_NAME_LENGTH:
        name = name[:MAX_NAME_LENGTH - 3].rstrip() + "..."
    return name


driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {
        "behavior": "allow",
        "downloadPath": os.path.abspath(PDF_BASE_DIR)
    }
)


for level_1, sublevels in structure.items():

    folder_1 = sanitize_name(level_1)

    for level_2, files in sublevels.items():

        folder_2 = sanitize_name(level_2)

        target_folder = os.path.join(PDF_BASE_DIR, folder_1, folder_2)

        try:
            os.makedirs(target_folder, exist_ok=True)
        except Exception:
            continue

        for file_info in files:

            try:
                url = file_info["url"]

                visible_name = sanitize_name(file_info["file_name"]) + ".pdf"
                final_path = os.path.join(target_folder, visible_name)

                if os.path.exists(final_path):
                    continue

                print(f"Descargando.... {final_path}")

                before_files = set(os.listdir(PDF_BASE_DIR))

                driver.get(url)

                start = time.time()
                downloaded_file = None

                while time.time() - start < DOWNLOAD_TIMEOUT:

                    current_files = set(os.listdir(PDF_BASE_DIR))
                    new_files = current_files - before_files

                    if new_files:

                        # ignore temporary chrome files
                        completed_files = [
                            f for f in new_files
                            if not f.endswith(".crdownload")
                        ]

                        if completed_files:
                            downloaded_file = completed_files[0]
                            break

                    time.sleep(1)

                if downloaded_file:

                    source_path = os.path.join(PDF_BASE_DIR, downloaded_file)

                    try:
                        os.replace(source_path, final_path)
                    except Exception:
                        continue

            except Exception:
                continue

Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO parte por abandono de destino del Cap....pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO Hoja de servicios del Cap. Sánchez Va....pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO Informe de Asesoría Jurídica General....pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\SECRETO oficio dando cuenta toma de declaración..pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\SECRETO oficio dando cuenta toma de declaración..pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO oficio dando cuenta toma de declaración..pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO 

KeyboardInterrupt: 

______________

## Prueba de descarga con URL específica

In [107]:
# PRUEBA

DOWNLOAD_TIMEOUT = 30

test_pdf_url = "https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/defensa/Causa_9481_Reservado_parte_por_abandono_de_destino_del_Cap_Sanchez_Valiente.pdf"

os.makedirs(PDF_BASE_DIR, exist_ok=True)

driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {
        "behavior": "allow",
        "downloadPath": os.path.abspath(PDF_BASE_DIR)
    }
)

print("Descargando PDF desde:", test_pdf_url)

before_files = set(os.listdir(PDF_BASE_DIR))

driver.get(test_pdf_url)

start = time.time()
downloaded_file = None

while time.time() - start < DOWNLOAD_TIMEOUT:

    current_files = set(os.listdir(PDF_BASE_DIR))
    new_files = current_files - before_files

    # Check only completed PDFs
    completed_pdfs = [
        f for f in new_files
        if f.lower().endswith(".pdf")
    ]

    if completed_pdfs:
        downloaded_file = completed_pdfs[0]
        break

    time.sleep(1)

if downloaded_file:
    print("Guardado PDF en:", os.path.abspath(os.path.join(PDF_BASE_DIR, downloaded_file)))
else:
    print("Timeout alcanzado. No se detectó descarga completa.")

Descargando PDF desde: https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/defensa/Causa_9481_Reservado_parte_por_abandono_de_destino_del_Cap_Sanchez_Valiente.pdf
Timeout alcanzado. No se detectó descarga completa.


In [56]:
driver.close()